<a href="https://colab.research.google.com/github/varaiitj2527/PRMLProject/blob/main/Resnet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/varaiitj2527/PRMLProject.git
%cd PRMLProject

Cloning into 'PRMLProject'...
remote: Enumerating objects: 81, done.
remote: Counting objects: 100% (63/63), done.
remote: Compressing objects: 100% (57/57), done.
remote: Total 81 (delta 16), reused 15 (delta 0), pack-reused 18 (from 2)
Receiving objects: 100% (81/81), 180.07 MiB | 23.65 MiB/s, done.
Resolving deltas: 100% (16/16), done.
Updating files: 100% (32/32), done.
Filtering content: 100% (12/12), 851.90 MiB | 74.05 MiB/s, done.
/content/PRMLProject


In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import pickle
import matplotlib.pyplot as plt

In [ ]:
def unpickle(filename) :

  with open(filename, 'rb') as file :
      data = pickle.load(file, encoding='bytes')

  return data


In [ ]:
def toPickle(data, filename) :

  with open(filename, 'wb') as file :
      pickle.dump(data, file)

In [ ]:
train_RawData = unpickle('PreProcessedData/RawPixels_train.pkl')
test_RawData = unpickle('PreProcessedData/RawPixels_test.pkl')

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import io


resnet = models.resnet50(pretrained=True)
resnet = nn.Sequential(*list(resnet.children())[:-1])
resnet.eval()

def extract_features(image_data, model):
    image = Image.fromarray(image_data)
    image = image.convert('RGB')
    preprocess = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    image = preprocess(image)
    image = image.unsqueeze(0)
    with torch.no_grad():
        features = model(image)
    features = features.squeeze(0).flatten()
    features_np = features.cpu().numpy()
    return features_np

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [ ]:
train_features = []

for i in train_RawData:
  features = extract_features(i, resnet)
  train_features.append(features)

toPickle(train_features, 'Resnet_train.pkl')

In [ ]:
print(len(train_features),train_features[0].shape[0])

50000 2048


In [ ]:
test_features = []

for i in test_RawData:
  features = extract_features(i, resnet)
  test_features.append(features)

toPickle(test_features, 'Resnet_test.pkl')

In [ ]:
print(len(test_features),test_features[0].shape[0])

10000 2048
